# DeepCT (2019)
---
[[paper]](https://arxiv.org/abs/1910.10687)<br>DeepCT = Deep Contextualized Term Weighting

DeepCT — это метод для улучшения классического информационного поиска (Information Retrieval), который использует BERT для перерасчета весов термов (токенов) внутри документа. Вместо того чтобы полагаться на простую частоту слова (TF), модель предсказывает значимость каждого слова в зависимости от его контекста.

__Idea:__ объединить глубокое понимание контекста трансформерами с эффективностью классического разреженного поиска (Sparse Retrieval). Вместо создания сложных векторных индексов (как в Dense Retrieval), мы просто подменяем значения TF в стандартном инвертированном индексе на «умные» веса, вычисленные нейросетью.

Предыдущие методы:
- **BM25 (1994)**: использует Term Frequency (TF) и Inverse Document Frequency (IDF). Проблема в том, что TF считает все вхождения слова одинаково важными, не учитывая, является ли слово ключевым для смысла конкретного абзаца.
- **Doc2Query (2019)**: генерирует возможные вопросы к документу и дописывает их в конец текста, увеличивая частоту важных слов. Это расширяет документ, но не меняет саму механику взвешивания существующих слов.
- **BERT Re-ranking (2018)**: использует BERT для сравнения запроса и документа. Метод очень точный, но слишком медленный для поиска по миллионам документов (First-stage retrieval), поэтому обычно применяется только для переранжирования топ-100 результатов.

### Постановка задачи
Дана коллекция документов. Необходимо построить такой индекс, который при использовании стандартных движков поиска (типа Lucene или Elasticsearch) будет выдавать более релевантные результаты за счет более точного определения веса каждого слова в контексте конкретного предложения или абзаца.

### Архитектура
DeepCT использует архитектуру **BERT-for-token-classification**:
- На вход подается текст документа (пассаж).
- BERT генерирует контекстные эмбеддинги для каждого токена.
- Поверх BERT накладывается простой линейный слой (Linear Layer), который предсказывает одно скалярное число для каждого токена — его **Contextual Term Importance**.

### Алгоритм обучения
Самая большая сложность — где взять Ground Truth для «важности» слова? Авторы предложили элегантное решение:
1. Берется датасет для Query-Passage Retrieval (например, MS MARCO), где известно, какие запросы относятся к каким документам.
2. Для каждого документа собираются все релевантные ему запросы из обучающей выборки.
3. Слово в документе считается важным, если оно часто встречается в запросах, по которым этот документ находят.
4. Целевое значение веса $y_i$ для токена $i$ рассчитывается как пропорция запросов, содержащих это слово.
5. Модель обучается минимизировать Mean Squared Error (MSE) между предсказанным весом и этим расчетным значением.

### Алгоритм инференса (Indexing)
Процесс применения модели происходит на этапе индексации, а не в момент запроса пользователя:
1. Прогнать все документы коллекции через обученный DeepCT.
2. Получить веса для каждого токена. Веса обычно масштабируются и округляются до целых чисел, чтобы имитировать частоту слова (TF).
3. Создать новый текст документа, где слова повторяются количество раз, пропорциональное их предсказанному весу (или просто записать веса в поле Payload инвертированного индекса).
4. Загрузить данные в стандартный поисковый движок (Elasticsearch/Lucene).
5. При поиске использовать обычный BM25, но вместо реального количества упоминаний слова использовать предсказанные веса.

### Результаты
- На датасете MS MARCO (поиск по пассажам) DeepCT показал Recall@1000 равный **92.9%**, в то время как классический BM25 достигал только **81.4%**. Это огромный скачок в 11.5 п.п. для первого этапа поиска.
- По метрике MRR@10 (качество ранжирования топ-10) модель превзошла BM25 на **25%** (с 0.190 до 0.243).
- Главное преимущество: скорость поиска остается такой же высокой, как у BM25 (миллисекунды), так как вся тяжелая работа нейросети выполняется заранее при построении индекса.

## 📝 Критический анализ

```markdown
# DeepCT (2019)
---
[[paper]](https://arxiv.org/abs/1910.10687)<br>DeepCT = Deep Contextualized Term Weighting

DeepCT — метод улучшения информационного поиска, использующий BERT для перерасчета весов термов в документе. Вместо частоты слова (TF) модель предсказывает значимость каждого слова в зависимости от контекста.

__Idea:__ объединить глубокое понимание контекста трансформерами с эффективностью Sparse Retrieval. Вместо сложных векторных индексов, заменяем TF на «умные» веса, вычисленные нейросетью.

Предыдущие методы:
- **BM25 (1994)**: использует TF и IDF, не учитывая контекст.
- **Doc2Query (2019)**: генерирует вопросы к документу, увеличивая частоту важных слов.
- **BERT Re-ranking (2018)**: точный, но медленный для поиска по миллионам документов.

### Постановка задачи
Создать индекс, который улучшит релевантность результатов поиска, используя стандартные движки (Lucene, Elasticsearch).

### Архитектура
DeepCT использует **BERT-for-token-classification**:
- Вход: текст документа.
- BERT генерирует эмбеддинги для каждого токена.
- Линейный слой предсказывает **Contextual Term Importance** для каждого токена.

<img src="img/img.png" width=500>

### Алгоритм обучения
1. Используется датасет для Query-Passage Retrieval (например, MS MARCO).
2. Собираются релевантные запросы для каждого документа.
3. Слово важно, если часто встречается в запросах.
4. Целевой вес $y_i$ токена $i$ — пропорция запросов с этим словом.
5. Модель минимизирует MSE между предсказанным и расчетным весом.

### Алгоритм инференса (Indexing)
1. Прогнать документы через обученный DeepCT.
2. Получить и масштабировать веса токенов.
3. Создать новый текст документа или записать веса в индекс.
4. Загрузить данные в поисковый движок.
5. Использовать BM25 с предсказанными весами.

### Результаты
- На MS MARCO DeepCT показал Recall@1000 **92.9%** против **81.4%** у BM25, улучшение на 11.5 п.п.
- MRR@10 улучшилось на **25%** (с 0.190 до 0.243).
- Скорость поиска остается на уровне BM25, так как нейросеть работает на этапе индексации.
```

## 💻 Пример кода

Иллюстративный Python пример, демонстрирующий основные концепции:

In [ ]:
import torch
from transformers import BertTokenizer, BertModel
import torch.nn as nn
import torch.nn.functional as F

# DeepCT Model: BERT for token classification with a linear layer on top
class DeepCTModel(nn.Module):
    def __init__(self, bert_model_name='bert-base-uncased'):
        super(DeepCTModel, self).__init__()
        self.bert = BertModel.from_pretrained(bert_model_name)
        # Linear layer to predict the contextual term importance
        self.linear = nn.Linear(self.bert.config.hidden_size, 1)

    def forward(self, input_ids, attention_mask):
        # Get contextual embeddings from BERT
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        sequence_output = outputs.last_hidden_state
        # Predict term importance for each token
        term_importance = self.linear(sequence_output).squeeze(-1)
        return term_importance

# Tokenizer for BERT
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

# Example document
document = "Deep learning models have revolutionized the field of artificial intelligence."

# Tokenize the document
inputs = tokenizer(document, return_tensors='pt', truncation=True, padding=True)

# Initialize the DeepCT model
model = DeepCTModel()

# Forward pass to get term importance scores
with torch.no_grad():
    term_importance_scores = model(inputs['input_ids'], inputs['attention_mask'])

# Convert scores to a more interpretable form (e.g., scale and round)
scaled_scores = F.softmax(term_importance_scores, dim=-1) * 100
rounded_scores = torch.round(scaled_scores)

# Display tokens with their importance scores
tokens = tokenizer.convert_ids_to_tokens(inputs['input_ids'][0])
for token, score in zip(tokens, rounded_scores[0]):
    print(f"Token: {token}, Importance Score: {score.item()}")

# Example of how to use these scores in a search index
# Here, we would typically repeat tokens based on their scores or store them in a payload
# For simplicity, let's just print the "expanded" document
expanded_document = []
for token, score in zip(tokens, rounded_scores[0]):
    expanded_document.extend([token] * int(score.item()))

print("Expanded Document for Indexing:", " ".join(expanded_document))
```

### Key Points:
1. **Model Architecture**: The `DeepCTModel` uses BERT for generating contextual embeddings and a linear layer to predict the importance of each token.
2. **Tokenization**: The document is tokenized using BERT's tokenizer, which is crucial for aligning input with the model's expectations.
3. **Inference**: The model predicts importance scores for each token, which are then scaled and rounded to simulate term frequency (TF).
4. **Indexing**: The tokens are repeated in the "expanded" document based on their importance scores, which can be used to create a more effective search index.
5. **Efficiency**: The heavy computation is done during indexing, allowing for fast retrieval using traditional search engines like Elasticsearch or Lucene.